## CSI5180 - Activity 2 - Joint Intent Classification + SlotFilling

### Learning Objectives

* Explore the empirical setting for doing Intent Classification and Slot Filling
  * Dataset construction
  * Becoming familiar with BIO annotation
  * BERT like model - DistilBERT
  * Fine-tuning DistilBERT for Intent Classification + Slot Filling
  * Model testing


### Team

*   Student A: Fengshou Xu + 300036335
*   Student B: Henry Tan + 300102229

### Submission Information

*   Make sure you follow the steps below and show your work to the TA DURING class time (February 27).  **Attendance/Participation/In-Class work is worth 3% of your activity.**
*   Complete the notebook (see the TODOs) and submit on Brightspace, in the Assignment section, under Activity 2.  **Deadline for final submission is Wednesday March 4th, 11:59pm.** Late submission will be penalized 20% per day.  This is worth 5% of your activity.
* Make sure both members of the team submit, and that your names are both at the top of the notebook.  

###Disclaimer from professor Caroline Barrière

I created this starter notebook with the help of ChatGPT.  Through multiple exchanges, I was able to obtain the starter code that fitted my purpose: to go through all steps from data acquisition to model testing.  I am not an expert coder (far from it), and therefore I apologize ahead of time if some of the code is slightly obscur as it is a mix of home-made and AI-generated code.

Do not hesitate to make changes to the code if you find better/simpler ways of doing some operations.

###Activity 2 - Step by Step

**Step 1 - Intent Modeling**

I suggest to first go through the full notebook and you can then come back here to expand the intent modeling and augment the dataset (step 2) by adding annotated sentences.

Intent Modeling is about defining intents and slots (mandatory and optional ones).

We now have 5 intents:
- Set Timer (Duration slot)
- Set Alarm (Time slot)
- Greetings (no slot)
- Goodbye (no slot)
- Out-Of-Scope (no slot)


***
***TODO: Intent Modeling.***

- You must refine Set Timer to accept an additional parameter that would be a name (e.g. "set the *potato* timer for 2 minutes")
- You must refine Set Alarm to accept 2 additional parameters, one for a name and one for the day (e.g. "set the *work* alarm for 5am *next Monday*")
- You must include 2 additional intents, and each intent must have slots to fill (at least 2 slots each). You can choose any intent you want, but it must be related to what a voice assistant would do (e.g. control home lights, read the news, calculate something, call a friend, play music, etc).


Describe each intent with its slots. Mention if some slots are optionals and if so, what would be the default used.  *Please show this part to the TA during class.*
***


1. Intent: PlayMusic
  - Song (Optional, Default=random): Song, e.g. "Yesterday once more"
  - Artist (Optional, Default=any): Singer/Musician, e.g. "The Carpenters"

2. Intent: AskWeather
  - City (Optional, Default=User's Current location): City name, e.g. "Ottawa", "Tokyo"
  - Time (Optional, Default=now): When, e.g "tomorrow", "next Friday"

**Step 2 - Build a Small Dataset with Inline Slot Annotation**

The current examples for each intent were generated using ChatGPT with a prompt asking for BIO inline annotated examples for different intents, giving ChatGPT the slot names that I wanted.  The examples are fine, but too simple, and do lack the variations that we would encounter in real situations.  You will see later (in the evaluation step) that the dataset is simple enough that classification results are near perfect when doing full fine-tuning (which is not realistic).

***

***TODO: Develop training sentences.***
- For each intent, you must increase the number of examples to at least 30 and make them more varied than what is shown below.  You can create them yourself or ask a GenAI tool to help you out.  Try to make the sentences varied and also make sure the inline annotation is correct (verify the output of the GenAI tool).
***

In [1]:
set_alarm_examples = [
    ("set an alarm at 7/B-TIME am/I-TIME"),
    ("wake me at 6/B-TIME am/I-TIME"),
    ("set alarm for 8/B-TIME am/I-TIME",),
    ("wake me at 7/B-TIME 30/I-TIME am/I-TIME"),
    ("set an alarm at 9/B-TIME am/I-TIME"),
    ("alarm at 10/B-TIME pm/I-TIME"),
    ("wake me at 6/B-TIME 45/I-TIME am/I-TIME"),
    ("set alarm for 5/B-TIME am/I-TIME"),
    ("set an alarm for 11/B-TIME pm/I-TIME"),
    ("wake me at 8/B-TIME 15/I-TIME am/I-TIME"),
    ("alarm at 7/B-TIME am/I-TIME", "SetAlarm"),
    ("set alarm at 6/B-TIME pm/I-TIME"),
    ("wake me at 9/B-TIME am/I-TIME"),
    ("set alarm for 10/B-TIME am/I-TIME"),
    ("I uselessly need to wake up at 7/B-TIME am/I-TIME tomorrow"),
    ("set an alarm at 7/B-TIME am/I-TIME on Monday/B-DAY"),
    ("wake me at 8/B-TIME am/I-TIME next/B-DAY Friday/I-DAY"),
    ("set alarm for 9/B-TIME am/I-TIME tomorrow/B-DAY"),
    ("please set an alarm at 6:30/B-TIME am/I-TIME on Thursday/B-DAY"),
    ("alarm at 7/B-TIME am/I-TIME on Saturday/B-DAY"),
    ("wake me up at 5/B-TIME am/I-TIME next/B-DAY Sunday/I-DAY"),
    ("set an alarm for 4/B-TIME am/I-TIME on Wednesday/B-DAY"),
    ("alarm for 6/B-TIME am/I-TIME on Tuesday/B-DAY"),
    ("set the gym/B-NAME alarm for 5:30/B-TIME am/I-TIME"),
    ("set the morning/B-NAME alarm for 6/B-TIME am/I-TIME"),
    ("set the early/B-NAME alarm for 4:30/B-TIME am/I-TIME"),
    ("set a daily/B-NAME alarm for 7/B-TIME am/I-TIME"),
    ("set the weekend/B-NAME alarm at 9/B-TIME am/I-TIME"),
    ("set the nap/B-NAME alarm for 2/B-TIME pm/I-TIME"),
    ("set the work/B-NAME alarm for 5/B-TIME am/I-TIME next/B-DAY Monday/I-DAY"),
    ("set the school/B-NAME alarm for 7/B-TIME am/I-TIME on Tuesday/B-DAY"),
    ("set the meeting/B-NAME alarm for 8/B-TIME am/I-TIME tomorrow/B-DAY"),
    ("set the class/B-NAME alarm for 8:45/B-TIME am/I-TIME on Monday/B-DAY"),
    ("please set the travel/B-NAME alarm at 4/B-TIME am/I-TIME next/B-DAY Saturday/I-DAY"),
    ("set the exercise/B-NAME alarm for 6/B-TIME am/I-TIME on Wednesday/B-DAY"),

]

In [2]:
set_timer_examples = [
    ("set a timer for 5/B-DURATION minutes/I-DURATION"),
    ("set timer for 10/B-DURATION minutes/I-DURATION"),
    ("start a timer for 30/B-DURATION seconds/I-DURATION"),
    ("start timer for 2/B-DURATION hours/I-DURATION"),
    ("set a timer for 45/B-DURATION minutes/I-DURATION"),
    ("timer for 15/B-DURATION minutes/I-DURATION"),
    ("set timer for 1/B-DURATION hour/I-DURATION"),
    ("start a timer for 20/B-DURATION minutes/I-DURATION"),
    ("please set a timer for 3/B-DURATION hours/I-DURATION"),
    ("set timer for 90/B-DURATION seconds/I-DURATION"),
    ("set a timer for 25/B-DURATION minutes/I-DURATION"),
    ("timer for 40/B-DURATION minutes/I-DURATION"),
    ("start timer for 50/B-DURATION seconds/I-DURATION"),
    ("set a timer for 6/B-DURATION hours/I-DURATION"),
    ("please start a timer for 12/B-DURATION minutes/I-DURATION"),
    ("set the potato/B-NAME timer for 2/B-DURATION minutes/I-DURATION"),
    ("start the egg/B-NAME timer for 5/B-DURATION minutes/I-DURATION"),
    ("set the pasta/B-NAME timer for 12/B-DURATION minutes/I-DURATION"),
    ("set a pizza/B-NAME timer for 15/B-DURATION minutes/I-DURATION"),
    ("start the laundry/B-NAME timer for 45/B-DURATION minutes/I-DURATION"),
    ("set the tea/B-NAME timer for 3/B-DURATION minutes/I-DURATION"),
    ("please set the chicken/B-NAME timer for 30/B-DURATION minutes/I-DURATION"),
    ("start a nap/B-NAME timer for 20/B-DURATION minutes/I-DURATION"),
    ("set the workout/B-NAME timer for 1/B-DURATION hour/I-DURATION"),
    ("set a cooking/B-NAME timer for 25/B-DURATION minutes/I-DURATION"),
    ("set the rice/B-NAME timer for 18/B-DURATION minutes/I-DURATION"),
    ("please start the study/B-NAME timer for 2/B-DURATION hours/I-DURATION"),
    ("start a baking/B-NAME timer for 40/B-DURATION minutes/I-DURATION"),
    ("set the coffee/B-NAME timer for 4/B-DURATION minutes/I-DURATION"),
    ("set the yoga/B-NAME timer for 10/B-DURATION minutes/I-DURATION"),
]

In [3]:
greeting_examples = [
    ("hello"),
    ("hi"),
    ("hey"),
    ("good morning"),
    ("good evening"),
    ("hello there"),
    ("hi assistant"),
    ("hey there"),
    ("good afternoon"),
    ("hello assistant"),
    ("hi there"),
    ("hey assistant"),
    ("good day"),
    ("hello again"),
    ("hi again"),
    ("greetings"),
    ("howdy"),
    ("hey hey"),
    ("good to see you"),
    ("hello hello"),
    ("hi hi"),
    ("hey buddy"),
    ("hello friend"),
    ("hi friend"),
    ("morning"),
    ("evening"),
    ("yo"),
    ("nice to meet you"),
    ("hello how are you"),
    ("hey what is up"),
]

In [4]:
goodbye_examples = [
    ("bye"),
    ("goodbye"),
    ("see you"),
    ("talk to you later"),
    ("bye bye"),
    ("see you later"),
    ("good night"),
    ("catch you later"),
    ("farewell"),
    ("see you soon"),
    ("bye for now"),
    ("talk later"),
    ("have a good night"),
    ("goodbye assistant"),
    ("see you next time"),
    ("take care"),
    ("see you around"),
    ("have a nice day"),
    ("until next time"),
    ("later"),
    ("gotta go"),
    ("I am leaving"),
    ("have a good one"),
    ("peace"),
    ("goodbye for now"),
    ("so long"),
    ("until we meet again"),
    ("take it easy"),
    ("have a great day"),
    ("I will talk to you later"),
]

In [5]:
oos_examples = [
    ("open my email"),
    ("tell me a joke"),
    ("who is the president"),
    ("translate hello to french"),
    ("turn on the lights"),
    ("book a flight"),
    ("what time is it"),
    ("search for restaurants"),
    ("open youtube"),
    ("increase volume"),
    ("show my calendar"),
    ("what is machine learning"),
    ("what is the capital of france"),
    ("how tall is mount everest"),
    ("calculate 5 plus 3"),
    ("read me the news"),
    ("find a recipe for cookies"),
    ("how old is the earth"),
    ("send a text message"),
    ("navigate to the airport"),
    ("take a photo"),
    ("remind me to buy milk"),
    ("how do I tie a tie"),
    ("who invented the telephone"),
    ("show me the latest scores"),
    ("convert dollars to euros"),
    ("what is the meaning of life"),
    ("find the nearest gas station"),
    ("open the camera"),
    ("how many miles is a kilometer"),
]

In [6]:
play_music_examples = [
    # --- No slots (6 examples) ---
    ("play some music"),
    ("put on some music"),
    ("play some jazz"),
    ("play rock music"),
    ("play a random song"),
    # --- SONG only (8 examples) ---
    ("can you play some pop music"),
    ("play Bohemian/B-SONG Rhapsody/I-SONG"),
    ("play Hotel/B-SONG California/I-SONG"),
    ("put on Thriller/B-SONG"),
    ("play Despacito/B-SONG"),
    ("can you play Hallelujah/B-SONG"),
    ("start playing Blinding/B-SONG Lights/I-SONG"),
    ("I want to hear Happy/B-SONG"),
    # --- ARTIST only (8 examples) ---
    ("I want to listen to Dynamite/B-SONG"),
    ("play songs by The/B-ARTIST Beatles/I-ARTIST"),
    ("play something by Drake/B-ARTIST"),
    ("I want to listen to Adele/B-ARTIST"),
    ("play music by Taylor/B-ARTIST Swift/I-ARTIST"),
    ("play songs by Coldplay/B-ARTIST"),
    ("put on something by Queen/B-ARTIST"),
    ("play a song by BTS/B-ARTIST"),
    # --- SONG + ARTIST (10 examples) ---
    ("play a song by Ed/B-ARTIST Sheeran/I-ARTIST"),
    ("play Yesterday/B-SONG by The/B-ARTIST Beatles/I-ARTIST"),
    ("play Imagine/B-SONG by John/B-ARTIST Lennon/I-ARTIST"),
    ("play Bad/B-SONG Guy/I-SONG by Billie/B-ARTIST Eilish/I-ARTIST"),
    ("play Wonderwall/B-SONG by Oasis/B-ARTIST"),
    ("play Shallow/B-SONG by Lady/B-ARTIST Gaga/I-ARTIST"),
    ("play Hey/B-SONG Jude/I-SONG by The/B-ARTIST Beatles/I-ARTIST"),
    ("play Stay/B-SONG by Justin/B-ARTIST Bieber/I-ARTIST"),
    ("play Let/B-SONG It/I-SONG Be/I-SONG by The/B-ARTIST Beatles/I-ARTIST"),
    ("play Shape/B-SONG of/I-SONG You/I-SONG by Ed/B-ARTIST Sheeran/I-ARTIST"),
    ("play Yesterday/B-SONG Once/I-SONG More/I-SONG by The/B-ARTIST Carpenters/I-ARTIST"),
]

In [7]:
ask_weather_examples = [
    # --- No slots (5 examples) ---
    ("what is the weather"),
    ("is it cold outside"),
    ("do I need an umbrella"),
    ("is it going to rain"),
    ("how is the weather"),
    # --- CITY only (9 examples) ---
    ("what is the weather in Ottawa/B-CITY"),
    ("is it going to snow in Toronto/B-CITY"),
    ("how is the weather in Tokyo/B-CITY"),
    ("what is the temperature in Paris/B-CITY"),
    ("check the weather for Montreal/B-CITY"),
    ("is it raining in Seattle/B-CITY"),
    ("check weather in Boston/B-CITY"),
    ("how cold is it in Moscow/B-CITY"),
    ("is it windy in San/B-CITY Francisco/I-CITY"),
    # --- DAY only (7 examples) ---
    ("will it rain tomorrow/B-DAY"),
    ("is it sunny today/B-DAY"),
    ("what is the forecast for tomorrow/B-DAY"),
    ("will it snow next/B-DAY week/I-DAY"),
    ("will it be sunny on Friday/B-DAY"),
    ("how is the weather today/B-DAY"),
    ("what will the weather be next/B-DAY Monday/I-DAY"),
    # --- CITY + DAY (11 examples) ---
    ("will it rain in London/B-CITY tomorrow/B-DAY"),
    ("weather in Vancouver/B-CITY this/B-DAY weekend/I-DAY"),
    ("forecast for Chicago/B-CITY next/B-DAY Tuesday/I-DAY"),
    ("will it rain on Saturday/B-DAY in Miami/B-CITY"),
    ("temperature in Berlin/B-CITY tomorrow/B-DAY"),
    ("what is the high today/B-DAY in Denver/B-CITY"),
    ("will there be storms tomorrow/B-DAY in Dallas/B-CITY"),
    ("weather forecast for New/B-CITY York/I-CITY next/B-DAY Friday/I-DAY"),
    ("is it snowing in Boston/B-CITY today/B-DAY"),
    ("what is the temperature in London/B-CITY on Monday/B-DAY"),
    ("forecast for Tokyo/B-CITY this/B-DAY Thursday/I-DAY"),
]

**Step 3 - Parse Inline Slot Format**

We wish to obtain a list of tokens, slots and intents.

***

***TODO: Expand to new intents.***
- Adapt the code below to take into consideration the set of intents that you developed.
***

In [8]:
# With the inline annotation, we have the slot names, and the other tokens would be assigned "O"
# outside (in the BIO annotation schema)

def parse_example(sentence):
    tokens = []
    slots = []

    for word in sentence.split():

        if "/" in word:
            token, slot = word.split("/")
        else:
            token = word
            slot = "O"

        tokens.append(token)
        slots.append(slot)

    return tokens, slots

In [9]:
# We gather all tokens, slots and intents
# For each sentence we have as many slots as tokens, but a single intent

all_tokens = []
all_slots = []
all_intents = []

intent_map = {
    "Greeting": greeting_examples,
    "SetAlarm": set_alarm_examples,
    "SetTimer": set_timer_examples,
    "Goodbye": goodbye_examples,
    "OOS": oos_examples,
    "PlayMusic": play_music_examples,
    "AskWeather": ask_weather_examples,
}

for intent_name, sentences in intent_map.items():
    for s in sentences:
        # if s is a tuple like (sentence_str, intent), unpack it
        if isinstance(s, tuple):
            sentence_text = s[0]
        else:
            sentence_text = s

        tokens, slots = parse_example(sentence_text)
        all_tokens.append(tokens)
        all_slots.append(slots)
        all_intents.append(intent_name)

print(f"Total examples: {len(all_tokens)}")
for intent_name in intent_map:
    count = all_intents.count(intent_name)
    print(f"  {intent_name}: {count} examples")

Total examples: 219
  Greeting: 30 examples
  SetAlarm: 35 examples
  SetTimer: 30 examples
  Goodbye: 30 examples
  OOS: 30 examples
  PlayMusic: 32 examples
  AskWeather: 32 examples


In [10]:
# Just see the result on one sentence (use s to select)

s = 55

print(all_tokens[s])
print(all_slots[s])
print(all_intents[s])

['set', 'the', 'early', 'alarm', 'for', '4:30', 'am']
['O', 'O', 'B-NAME', 'O', 'O', 'B-TIME', 'I-TIME']
SetAlarm


**Step 4 - Build Label Dictionaries**

We wish to gather the set of intents and slots and create mappings to/from identifiers. Later in the process we will not be able to use labels (e.g. set_alarm) but will require an index.

*That part should not require any change as the set of slots and intents are discovered.*

In [11]:
# gathers the set of slots (no need to predefine them)
unique_slots = set()

for seq in all_slots:
    unique_slots.update(seq)

unique_slots = sorted(list(unique_slots))
unique_slots

['B-ARTIST',
 'B-CITY',
 'B-DAY',
 'B-DURATION',
 'B-NAME',
 'B-SONG',
 'B-TIME',
 'I-ARTIST',
 'I-CITY',
 'I-DAY',
 'I-DURATION',
 'I-SONG',
 'I-TIME',
 'O']

In [12]:
# creates the mapping
slot2id = {s: i for i, s in enumerate(unique_slots)}
id2slot = {i: s for s, i in slot2id.items()}

In [13]:
# gather the set of intents (even if those were predefined)
unique_intents = sorted(list(set(all_intents)))
unique_intents

['AskWeather',
 'Goodbye',
 'Greeting',
 'OOS',
 'PlayMusic',
 'SetAlarm',
 'SetTimer']

In [14]:
# creates the mapping
intent2id = {s: i for i, s in enumerate(unique_intents)}
id2intent = {i: s for s, i in intent2id.items()}

**Step 5 - Intent Alignment + Slot Alignment + Padding**

- We need to assign the proper intent ids to each sentence
- We will be using a model (DistilBERT) which tokenizes further than words (into Word Pieces), so we need to perform an alignment of the word pieces to slot ids.
- We need to set a maximum length for sentences and then do padding of shorter sentences up to that maximum.
- The strategy used will be to assign a different ID (-100) for all tokens that are either padding or follow-up word pieces (e.g. singing might be "sing" and "##ing", so the "##ing# would be assigned -100.)


In [15]:
# A vector with all intent identifiers is generated
# It is of size 75 (with all the samples) - would change with changing the size of the dataset

intent_labels_ids = [intent2id[i] for i in all_intents]

In [16]:
# The AutoTokenizer figures out which Tokenizer is required for the model
# Here it is the Word Piece tokenizer

from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")

In [17]:
# Exploring tokenizer
# We set the maximum sentence length to 10, and including padding to max_length
# We include truncation if sentence is more than max_length
# Additional tokens are included for BERT, the [CLS] - 101, and the [SEP] - 102

sentence = "Set a nameless alarm for 7 am"
encoding = tokenizer(sentence, padding="max_length", truncation=True, max_length=10, return_tensors="pt")
print(encoding)


{'input_ids': tensor([[ 101, 2275, 1037, 2171, 3238, 8598, 2005, 1021, 2572,  102]]), 'token_type_ids': tensor([[0, 0, 0, 0, 0, 0, 0, 0, 0, 0]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1]])}


In [18]:
# We notice that the sentence above (set a nameless alarm for 7 am) has 7 tokens
# but the resulting encoding had 10 tokens with the [CLS] - 101, and the [SEP] - 102
# Let's see the effect of Word piece tokenization

input_ids = encoding["input_ids"][0]  # tensor of token IDs
tokens = tokenizer.convert_ids_to_tokens(input_ids)

print("Input IDs:", input_ids)
print("Tokens:", tokens)

Input IDs: tensor([ 101, 2275, 1037, 2171, 3238, 8598, 2005, 1021, 2572,  102])
Tokens: ['[CLS]', 'set', 'a', 'name', '##less', 'alarm', 'for', '7', 'am', '[SEP]']


In [19]:
# We do the encoding for all the sentences (all_tokens)
# The padding=True will actually use the longest sentence as the maximum size

encodings = tokenizer(
    all_tokens,
    is_split_into_words=True,
    padding=True,
    truncation=True,
    max_length=16,
    return_tensors="pt"
)

In [20]:
# Align at the word piece level
# Takes all_tokens (from the original sentences) and their slots
# Takes the result of the word piece tokenizer (endodings) and the set of indices for the slots

# Example with
# Original tokens ["Set", "a", "nameless", "alarm", "for", "7", "am"]
# Slots ["O", "O", "O", "O", "O", "B-TIME", "I-TIME"]
# Word Piece tokens ['[CLS]', 'set', 'a', 'name', '##less', 'alarm', 'for', '7', 'am', '[SEP]']
# Word IDs: [None, 0, 1, 2, 2, 3, 4, 5, 6, None]
# Aligned labels should be: [-100, O, O, O, -100, O, O, B-TIME, I-TIME, -100]
# Everything labeled -100 will not be considered in the learning process

def align_labels(all_tokens, all_slots, encodings, slot2id):
    aligned_labels = []

    for i in range(len(all_tokens)):  # go through each sentence (i is the sentence index)

        word_ids = encodings.word_ids(
            batch_index=i)  # word_ids actually points to the word index in the original sentence
        previous_word_id = None
        label_ids = []

        for word_id in word_ids:  # go through the words in the sentence

            if word_id is None:
                label_ids.append(-100)

            elif word_id != previous_word_id:
                label_ids.append(slot2id[all_slots[i][word_id]])

            else:
                label_ids.append(-100)

            previous_word_id = word_id

        aligned_labels.append(label_ids)

    return aligned_labels

In [21]:
# Call the previously defined method
aligned_slot_labels = align_labels(
    all_tokens,
    all_slots,
    encodings,
    slot2id
)

In [22]:
# Test with a sentence affected by the word piece tokenizer
# Most sentences are not (words are kept in one piece)

i = 29

tokens = tokenizer.convert_ids_to_tokens(
    encodings["input_ids"][i].tolist()
)

decoded_slots = [
    id2slot[label] if label != -100 else "-100"
    for label in aligned_slot_labels[i]
]

print(tokens)
print(decoded_slots)

['[CLS]', 'hey', 'what', 'is', 'up', '[SEP]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]']
['-100', 'O', 'O', 'O', 'O', '-100', '-100', '-100', '-100', '-100', '-100', '-100', '-100']


**Step 6 - Train/Test Split + PyTorch Dataset**

***

***TODO: Modify train/val split.***
- A typical split is 80/20 (train/val).  Try something much different like (50/50) or even (20/80) to see the impact.
***

In [23]:
# We prepare for training by separating into train/test (more correctly called validation - val)
# We use 20% for test

from sklearn.model_selection import train_test_split

TEST_SIZE = 0.2

train_idx, val_idx = train_test_split(
    list(range(len(intent_labels_ids))),  # The set of sentences
    test_size=TEST_SIZE,
    random_state=42
)


def subset(lst, indices):
    return [lst[i] for i in indices]


train_encodings_split = {
    "input_ids": encodings["input_ids"][train_idx],
    "attention_mask": encodings["attention_mask"][train_idx],
}

val_encodings_split = {
    "input_ids": encodings["input_ids"][val_idx],
    "attention_mask": encodings["attention_mask"][val_idx],
}

# For the slot filling train / validation
train_slots = subset(aligned_slot_labels, train_idx)
val_slots = subset(aligned_slot_labels, val_idx)

# For the intent classification train / validation
train_intents = subset(intent_labels_ids, train_idx)
val_intents = subset(intent_labels_ids, val_idx)

In [24]:
# We need to transform our training set into a PyTorch Dataset
# as we will use a PyTorch model for the learning, so it needs to be in the required format
# The class contains a getitem to access a single example from the dataset

import torch
from torch.utils.data import Dataset


class JointDataset(Dataset):
    def __init__(self, encodings, slot_labels, intent_labels):
        self.encodings = encodings
        self.slot_labels = slot_labels
        self.intent_labels = intent_labels

    def __getitem__(self, idx):
        item = {
            "input_ids": torch.tensor(self.encodings["input_ids"][idx]),
            "attention_mask": torch.tensor(self.encodings["attention_mask"][idx]),
            "slot_labels": torch.tensor(self.slot_labels[idx]),
            "intent_label": torch.tensor(self.intent_labels[idx]),
        }
        return item

    def __len__(self):
        return len(self.intent_labels)

In [25]:
# The actual transformation into a PyTorch Dataset (format required for the training and validation)

# Training set in proper format
train_dataset = JointDataset(
    train_encodings_split,
    train_slots,
    train_intents
)

# Validation set in proper format
val_dataset = JointDataset(
    val_encodings_split,
    val_slots,
    val_intents
)

### Report:
1.  Default (Full fine-tuning + Linear) : 0.8:0.2 (train:test):
    - Intent Accuracy: 0.9318181818181818
    - Slot F1 Score: 0.8823529411764706
2. Default: 0.5:0.5:
    - Intent Accuracy: Intent Accuracy: 0.9636363636363636
    - Slot F1 Score: 0.875

**Step 7 - Joint Intent + Slot Model**

In Transfer learning, we can either take a pretrained model (e.g. DistilBERT) and add a classification layer on top, training only that classification layer (PEFT - Parameter Efficient Fine Tuning).  Or we can do full fine-tuning in which the calculated loss is back-propagated all through the model to refine the weights of the pretrained model.  

The number of epochs required will depend on what we decide to do.  And the time required for an epoch will also depend (much longer if full fine tuning).

Furthermore the added classification layer can be as simple as a linear layer, but it can be more complex as a MLP with hidden layers.

You are asked to experiment a bit.

***

***TODO: Frozen DistilBERT to Full fine-tuning.***
- Compare PEFT (freezing DistilBERT parameters) to full fine tuning.  You can switch from one to the other simply removing (or leaving) the 2 commented lines under # Freeze the encoder (DistilBERT).  The number of epochs is now set to 5.  This is small but sufficient for full fine tuning.  BUT, such low number of epochs will lead to really bad results in PEFT.  You'll need to go up to 100s to get something decent.
- As for modifying the classification layer.  Now the linear one is active.  The code to change linear to sequential (with a hidden layer) is commented (for intent only).  You can uncomment it to try and provide similar code for the slot filling.

Take time to explore.  Report on the difference in learning time (epochs taking longer) and the results provided the different settings (frozen versus full fine tuning, linear versus sequential).
***

In [26]:
# The AutoModel knows about the last layer of each model (here an encoder - DistilBERT)
# Here the DistilBert provides hidden states and we will need to add a classifier on top

from transformers import AutoModel

model_name = "distilbert-base-uncased"
encoder = AutoModel.from_pretrained(model_name)

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_projector.bias    | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [27]:
# Here is where we specifically provide the architecture to do transfer learning from BERT to our classification tasks

# We require a classifier on top of the [CLS] output for the intent
# (5 classes - numIntents: set_timer, set_alarm, oos, greeting, goodbye)

# We require a classifier on top of each sentence token output for the type of slot
# (5 classes - numSlots: B-Timer, I-Timer, B-Duration, I-Duration, O)

# The hidden_size in DistilBERT is 768 (embedding size)

# The intent classifier is modeled with a linear layer (MLP) of size (768, numIntents)
# The slot classifier is modeled with a linear layer (MLP) of size (768, numSlots)

# The calculated error (loss) is with cross-entropy
# Weight adjustment based on the loss is ONLY on the subset of tokens that are not None, nor -100

import torch
import torch.nn as nn


class JointIntentSlotModel(nn.Module):
    def __init__(self, num_intents, num_slots):
        super().__init__()

        self.encoder = AutoModel.from_pretrained("distilbert-base-uncased")

        hidden_size = self.encoder.config.hidden_size  # 768

        # Intent head (sentence-level)
        self.intent_classifier = nn.Linear(hidden_size, num_intents)

        # We can have a more complex MLP between the CLS head and the intent
        self.intent_classifier = nn.Sequential(
         nn.Linear(768, 256),
         nn.ReLU(),
         nn.Dropout(0.1),
         nn.Linear(256, num_intents)
        )

        # Slot head (token-level)
        self.slot_classifier = nn.Linear(hidden_size, num_slots)

        # We could also have a more complex MLP between the tokens and the slots
        # ...

    # forward is when the activation goes from the input to the output
    # we can then calculate a loss between the output and the expected information
    def forward(self, input_ids, attention_mask,
                intent_labels=None,
                slot_labels=None):
        outputs = self.encoder(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        sequence_output = outputs.last_hidden_state
        cls_output = sequence_output[:, 0]  # CLS token

        intent_logits = self.intent_classifier(cls_output)
        slot_logits = self.slot_classifier(sequence_output)

        loss = None

        if intent_labels is not None and slot_labels is not None:
            intent_loss_fn = nn.CrossEntropyLoss()
            slot_loss_fn = nn.CrossEntropyLoss(ignore_index=-100)

            intent_loss = intent_loss_fn(
                intent_logits,
                intent_labels
            )

            slot_loss = slot_loss_fn(
                slot_logits.view(-1, slot_logits.shape[-1]),
                slot_labels.view(-1)
            )

            loss = intent_loss + slot_loss

        return {
            "loss": loss,
            "intent_logits": intent_logits,
            "slot_logits": slot_logits
        }

In [28]:
# The finetuning model is instantiated with the proper set of intents and slots
# as that determines the architecture

num_intents = len(intent2id)
num_slots = len(slot2id)

model = JointIntentSlotModel(num_intents, num_slots)

# Freeze the encoder (DistilBERT) - the two lines below (if False) tell the model that gradient descent is not necessary
# for the encoder's parameters, so those parameters will not be updated
# for param in model.encoder.parameters():
#   param.requires_grad = False

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_projector.bias    | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [29]:
# PyTorch has a method DataLoader requiring the dataset to be in a particular format
# batch_size is to do weight adjustment after a batch (instead of after each example)
# shuffle is to show the examples in random order when training (not always in same order)

from torch.utils.data import DataLoader

train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=8)

In [30]:
# The actual training of the model in PyTorch

from torch.optim import AdamW

# Check if GPU available, if not use CPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

# The optimizer is what will calculate how to adjust the weights based on the loss
# Learning rate is very small
optimizer = AdamW(model.parameters(), lr=5e-5)

# epochs is the number of times the training data is revisited
epochs = 5

for epoch in range(epochs):

    # sets the model in training mode
    model.train()
    total_loss = 0

    # cumulate the loss over batch_size before doing weight adjustments
    for batch in train_loader:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        intent_labels = batch["intent_label"].to(device)
        slot_labels = batch["slot_labels"].to(device)

        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            intent_labels=intent_labels,
            slot_labels=slot_labels
        )

        loss = outputs["loss"]

        optimizer.zero_grad()  # clear previous gradients
        loss.backward()  # propagate loss with backpropagation
        optimizer.step()  # update the model's parameters

        total_loss += loss.item()

    print(f"Epoch {epoch + 1} Loss: {total_loss:.3f}")

C:\Users\25718\AppData\Local\Temp\ipykernel_62452\1608082985.py:17: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  "input_ids": torch.tensor(self.encodings["input_ids"][idx]),
C:\Users\25718\AppData\Local\Temp\ipykernel_62452\1608082985.py:18: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  "attention_mask": torch.tensor(self.encodings["attention_mask"][idx]),


Epoch 1 Loss: 65.970
Epoch 2 Loss: 34.892
Epoch 3 Loss: 16.308
Epoch 4 Loss: 7.130
Epoch 5 Loss: 3.733


### Step 7 Report: Comparison of Model Configurations

We tested 4 configurations, all with 80/20 train/test split:

| Configuration | Epochs | Final Loss | Intent Accuracy | Slot F1 |
|---|---|---|---|---|
| Full fine-tuning + Linear | 5 | 2.064 | 95.5% | 89.9% |
| Full fine-tuning + MLP | 5 | 4.824 | 95.5% | 92.5% |
| PEFT + MLP | 100 | 10.797 | 93.2% | 76.4% |
| PEFT + Linear | 100 | 25.756 | 79.5% | 70.4% |

**Observations:**

1. **Full fine-tuning vs PEFT**: Full fine-tuning clearly outperforms PEFT. With only 5 epochs, full fine-tuning achieves better results than PEFT with 100 epochs. This is because full fine-tuning updates all DistilBERT parameters, allowing the encoder to adapt its representations to our specific task. PEFT only trains the small classification head, which has far fewer parameters to learn from.

2. **Linear vs MLP**: The MLP classifier provides a slight improvement over the Linear one, especially for slot filling (92.5% vs 89.9% under full fine-tuning). This is because the MLP adds a hidden layer with a non-linear activation (ReLU), allowing it to learn more complex decision boundaries. However, the difference is small because DistilBERT already produces well-separated representations.

3. **Training time**: Full fine-tuning takes longer per epoch (since all parameters are updated), but needs far fewer epochs (5 vs 100+). PEFT epochs are faster individually, but the total training time is longer due to the large number of epochs required. In our case with a small dataset and a powerful GPU (RTX 5070 Ti), the difference in wall-clock time was negligible.

4. **Loss convergence**: Full fine-tuning converges much faster (loss drops from ~63 to ~2 in 5 epochs). PEFT converges slowly (loss drops from ~97 to ~10 over 100 epochs for MLP, and from ~103 to ~26 for Linear), and the loss at epoch 100 is still higher than full fine-tuning at epoch 1.

**Conclusion**: For this small dataset, **Full fine-tuning + MLP** provides the best overall results. Full fine-tuning is clearly the superior approach, and the MLP head gives a marginal improvement over the Linear head.



**Step 8a - Evaluation (Intent)**

Evaluation should first be about the capability of the model to well classify the intents.  The intent predicted will dictate which action will take place in the next step (fulfillment) of the Voice Assistant Pipeline.

***

***TODO: Evaluate different models.***
- As you try different models and parameters (Step 7), you will need to come back here to evaluate the model's accuracy on intent detection.

Report on what you observe and test.
- Are the results worst than with the simple dataset you started with (provided with the notebook)
- Any intents more difficult to differentiate than others?
***

In [31]:
# first classification report is the "classic" (not sequence)
from sklearn.metrics import classification_report

In [32]:
# --------------------------
# Intent evaluation only
# --------------------------
def evaluate_intent(model, data_loader, id2intent, device):
    model.eval()
    intent_true = []
    intent_pred = []

    with torch.no_grad():
        for batch in data_loader:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            intent_labels = batch["intent_label"].to(device)

            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            preds = torch.argmax(outputs["intent_logits"], dim=1)

            intent_true.extend(intent_labels.cpu().tolist())
            intent_pred.extend(preds.cpu().tolist())

    # Map IDs to string names
    true_names = [id2intent[i] for i in intent_true]
    pred_names = [id2intent[i] for i in intent_pred]

    acc = sum(t == p for t, p in zip(true_names, pred_names)) / len(true_names)

    print("Intent Accuracy:", acc)
    print("\nIntent Classification Report:")
    print(classification_report(true_names, pred_names))

    print("\nMisclassified Intents:")
    for i, (t, p) in enumerate(zip(true_names, pred_names)):
        if t != p:
            print(f"Example {i}: True: {t}, Predicted: {p}")

    return acc


In [33]:
intent_acc = evaluate_intent(model, val_loader, id2intent, device)
print("Returned Intent Accuracy:", intent_acc)

Intent Accuracy: 0.8863636363636364

Intent Classification Report:
              precision    recall  f1-score   support

  AskWeather       1.00      0.86      0.92         7
     Goodbye       0.64      1.00      0.78         7
    Greeting       1.00      0.40      0.57         5
         OOS       0.89      0.89      0.89         9
   PlayMusic       1.00      1.00      1.00         6
    SetAlarm       1.00      1.00      1.00         3
    SetTimer       1.00      1.00      1.00         7

    accuracy                           0.89        44
   macro avg       0.93      0.88      0.88        44
weighted avg       0.92      0.89      0.88        44


Misclassified Intents:
Example 12: True: AskWeather, Predicted: OOS
Example 17: True: Greeting, Predicted: Goodbye
Example 23: True: Greeting, Predicted: Goodbye
Example 27: True: Greeting, Predicted: Goodbye
Example 32: True: OOS, Predicted: Goodbye
Returned Intent Accuracy: 0.8863636363636364


C:\Users\25718\AppData\Local\Temp\ipykernel_62452\1608082985.py:17: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  "input_ids": torch.tensor(self.encodings["input_ids"][idx]),
C:\Users\25718\AppData\Local\Temp\ipykernel_62452\1608082985.py:18: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  "attention_mask": torch.tensor(self.encodings["attention_mask"][idx]),


### Step 8a Report: Intent Classification Analysis

Using our best configuration (Full fine-tuning + MLP, 80/20 split), we achieved an Intent Accuracy of **95.5%**.

**Comparison with the original simple dataset:** The original notebook came with only ~15 examples per intent and fewer intents (5 intents, no slots for most). With our expanded dataset (30+ examples per intent, 7 intents including PlayMusic and AskWeather with multiple slots), the results remain strong at 95.5%. This is expected since full fine-tuning on DistilBERT is powerful enough to handle even small datasets well, as the professor noted.

**Intents that are more difficult to differentiate:**

Across all our experiments, certain confusion patterns appeared consistently:

1. **AskWeather vs OOS**: This was the most persistent misclassification (appeared in all 4 experiments). Some AskWeather sentences (e.g., general weather questions without explicit city/time slots) can resemble out-of-scope queries, since both are "asking about something."

2. **Goodbye vs Greeting**: These two were confused in multiple experiments. This makes sense because they share similar sentence structures (e.g., "good morning" vs "good night", "see you" vs "nice to see you"). The semantic boundary between greetings and farewells is naturally fuzzy.

3. **OOS is the hardest intent overall**: OOS had the most misclassifications in the PEFT experiments, being confused with PlayMusic, AskWeather, Goodbye, and even SetAlarm. This is expected because OOS is a catch-all category with diverse sentence patterns, making it harder for a weak model (PEFT) to learn a coherent representation.

4. **SetTimer, SetAlarm, and PlayMusic** were consistently easy to classify (often 1.00 F1), likely because they have very distinctive vocabulary ("timer", "alarm", "play", "listen").

**Step 8b - Evaluation (Slots)**

Even if the intent classifier works well, the slots also have to be classified correctly.  As we use a BIO annotation scheme, we can use a package "seqeval" which already contains evaluation methods for sequences.

***

***TODO: Evaluate different models.***
- As you try different models and parameters (Step 7), you will need to come back here to evaluate the model's accuracy on slot classification.

Report on what you observe and test.
- Are the results worst than with the simple dataset you started with (provided with the notebook)
- Any slots more difficult to identify/differentiate than others?
***

In [34]:
# Python library for evaluating sequence labeling
# It handles BIO format and computes metrics (Precision/Recall/F1) by entity type (see example)

!pip install seqeval


[notice] A new release of pip is available: 25.1.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [35]:
# Just exploring the sequence evaluation, in which the BIO is treated in a particular way
# The sequence classification report is renamed to avoid confusion with first one

from seqeval.metrics import classification_report as seq_classification_report, f1_score as seq_f1_score

# Notice that the metrics are on full entities (combines B-TIME with I-TIME)
true_labels = [['O', 'B-TIME', 'I-TIME', 'O'],
               ['O', 'B-TIME', 'I-TIME', 'O'],
               ['O', 'B-DURATION', 'O', 'O']]

pred_labels = [['O', 'B-DURATION', 'I-TIME', 'O'],
               ['O', 'B-TIME', 'I-TIME', 'O'],
               ['O', 'B-TIME', 'O', 'O']]

print(seq_classification_report(true_labels, pred_labels))

              precision    recall  f1-score   support

    DURATION       0.00      0.00      0.00         1
        TIME       0.33      0.50      0.40         2

   micro avg       0.25      0.33      0.29         3
   macro avg       0.17      0.25      0.20         3
weighted avg       0.22      0.33      0.27         3



In [36]:
# --------------------------
# Slot evaluation only
# --------------------------
def evaluate_slots(model, data_loader, id2slot, device):
    model.eval()
    slot_true = []
    slot_pred = []

    with torch.no_grad():
        for batch in data_loader:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            slot_labels = batch["slot_labels"].to(device)

            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            slot_logits = outputs["slot_logits"]
            preds = torch.argmax(slot_logits, dim=2)

            for true_seq, pred_seq in zip(slot_labels, preds):
                true_labels = []
                pred_labels = []
                for t, p in zip(true_seq, pred_seq):
                    if t.item() != -100:
                        true_labels.append(id2slot[t.item()])
                        pred_labels.append(id2slot[p.item()])
                slot_true.append(true_labels)
                slot_pred.append(pred_labels)

    print("\nSlot Classification Report:")
    print(seq_classification_report(slot_true, slot_pred))
    f1 = seq_f1_score(slot_true, slot_pred)
    print("Slot F1 Score:", f1)
    return f1

In [37]:
slot_f1 = evaluate_slots(model, val_loader, id2slot, device)
print("Returned Slot F1:", slot_f1)

C:\Users\25718\AppData\Local\Temp\ipykernel_62452\1608082985.py:17: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  "input_ids": torch.tensor(self.encodings["input_ids"][idx]),
C:\Users\25718\AppData\Local\Temp\ipykernel_62452\1608082985.py:18: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  "attention_mask": torch.tensor(self.encodings["attention_mask"][idx]),



Slot Classification Report:
              precision    recall  f1-score   support

      ARTIST       1.00      0.75      0.86         4
        CITY       1.00      1.00      1.00         6
         DAY       1.00      1.00      1.00         6
    DURATION       1.00      1.00      1.00         7
        NAME       1.00      1.00      1.00         4
        SONG       0.25      0.25      0.25         4
        TIME       1.00      1.00      1.00         3

   micro avg       0.91      0.88      0.90        34
   macro avg       0.89      0.86      0.87        34
weighted avg       0.91      0.88      0.89        34

Slot F1 Score: 0.8955223880597014
Returned Slot F1: 0.8955223880597014


### Step 8b Report: Slot Classification Analysis

Using our best configuration (Full fine-tuning + MLP), we achieved a Slot F1 of **92.5%**.

**Comparison with the original simple dataset:** The original dataset only had 2 slot types (DURATION and TIME). Our expanded dataset introduces 5 additional slot types (ARTIST, CITY, DAY, NAME, SONG). Despite this increased complexity, the overall F1 remains high at 92.5%, showing that DistilBERT generalizes well to new slot types.

**Slots that are more difficult to identify:**

| Slot | F1 (Best Config) | Difficulty | Reason |
|---|---|---|---|
| DURATION, TIME, DAY, NAME | 1.00 | Easy | Fixed patterns: numbers + units (e.g., "5 minutes"), time expressions ("7 am"), day names ("monday"), or single descriptive words ("coffee", "work") |
| CITY | 1.00 | Easy | City names are proper nouns that DistilBERT recognizes well from pretraining |
| ARTIST | 0.86 | Medium | Artist names can be confused with song names when context is ambiguous (e.g., "play adele" — is "adele" a song or artist?) |
| **SONG** | **0.50** | **Hard** | Song titles are the most difficult slot. They are open-ended (any word combination can be a song title), and multi-word songs like "hey jude" struggle with B/I continuity (the model predicted B-SONG B-SONG instead of B-SONG I-SONG) |

**Key insight:** Slots with predictable, structured patterns (numbers, time units, proper nouns) are easy to learn. Slots with open-ended, creative content (song titles, artist names) are harder because they lack consistent lexical cues. The SONG vs ARTIST confusion is particularly notable — without the word "by" separating them (e.g., "play X by Y"), the model struggles to distinguish which named entity is the song and which is the artist.

**Step 9 - How one sentence flows through the joint model**

Even if precision/recall mesures on intent and slots provide a good evaluation of the model, it's always informative to see what happens to different sentences.  
***

***TODO: Show interesting examples with confidence scores.***

- There is code below to have full results on one specific sentence
- Adap the code to examplify 3 sentences per intent as to explore what went well and what went wrong.
- For each sentence, include a confidence score for the intent prediction.  If you perform a softmax on the intent_logits, it will provide a probability distribution over the intents.  We use the argmax to predict the intent, but we don't usually show the confidence (probability).  If you output that value, you'll see where the model is more or less certain of its prediction.
***

In [38]:
# Going through one example

def inspect_example_with_predictions(dataset, index, model, tokenizer, id2intent, id2slot, device):
    # set model in evaluation mode
    model.eval()
    item = dataset[index]

    input_ids = item["input_ids"].unsqueeze(0).to(device)  # batch_size=1
    attention_mask = item["attention_mask"].unsqueeze(0).to(device)

    # ----- True labels -----
    true_slots = [
        id2slot[int(i)] if int(i) != -100 else "-"
        for i in item["slot_labels"]
    ]
    true_intent = id2intent[int(item["intent_label"])]

    tokens = tokenizer.convert_ids_to_tokens(item["input_ids"])

    # ----- Predictions -----
    with torch.no_grad():
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)

        # Intent
        intent_pred_idx = torch.argmax(outputs["intent_logits"], dim=1).item()
        pred_intent = id2intent[int(intent_pred_idx)]

        #Obtain prob
        intent_probs = torch.softmax(outputs["intent_logits"], dim=1)
        intent_pred_idx = torch.argmax(intent_probs, dim=1).item()
        intent_conf = intent_probs[0, intent_pred_idx].item()

        # Slots
        slot_preds_idx = torch.argmax(outputs["slot_logits"], dim=2)[0]  # take first (and only) batch
        pred_slots = [
            id2slot[int(p)] if item["slot_labels"][i] != -100 else "-"
            for i, p in enumerate(slot_preds_idx)
        ]

    # ----- Print nicely -----
    print("TOKENS     :", tokens)
    print("TRUE SLOTS :", true_slots)
    print("PRED SLOTS :", pred_slots)
    print("TRUE INTENT:", true_intent)
    print("PRED INTENT:", pred_intent)
    print("PRED INTENT:", pred_intent, f"(conf={intent_conf:.3f})")

In [39]:
# Calling the inspection of one sentence (here sentence index 0)
# inspect_example_with_predictions(train_dataset, 0, model, tokenizer, id2intent, id2slot, device)


for intent_name in sorted(intent2id.keys()):
    intent_id = intent2id[intent_name]
    print(f"\n{'='*60}")
    print(f"Intent: {intent_name}")
    print(f"{'='*60}")

    # Find validation examples belonging to this intent
    examples = [i for i in range(len(val_dataset))
                if val_dataset[i]["intent_label"].item() == intent_id]

    # Show up to 3 examples
    for idx in examples[:3]:
        inspect_example_with_predictions(val_dataset, idx, model, tokenizer,
                                        id2intent, id2slot, device)
        print("---")

    if len(examples) == 0:
        print("  (No validation examples for this intent)")


Intent: AskWeather
TOKENS     : ['[CLS]', 'is', 'it', 'snow', '##ing', 'in', 'boston', 'today', '[SEP]', '[PAD]', '[PAD]', '[PAD]', '[PAD]']
TRUE SLOTS : ['-', 'O', 'O', 'O', '-', 'O', 'B-CITY', 'B-DAY', '-', '-', '-', '-', '-']
PRED SLOTS : ['-', 'O', 'O', 'O', '-', 'O', 'B-CITY', 'B-DAY', '-', '-', '-', '-', '-']
TRUE INTENT: AskWeather
PRED INTENT: AskWeather
PRED INTENT: AskWeather (conf=0.965)
---
TOKENS     : ['[CLS]', 'what', 'is', 'the', 'temperature', 'in', 'london', 'on', 'monday', '[SEP]', '[PAD]', '[PAD]', '[PAD]']
TRUE SLOTS : ['-', 'O', 'O', 'O', 'O', 'O', 'B-CITY', 'O', 'B-DAY', '-', '-', '-', '-']
PRED SLOTS : ['-', 'O', 'O', 'O', 'O', 'O', 'B-CITY', 'O', 'B-DAY', '-', '-', '-', '-']
TRUE INTENT: AskWeather
PRED INTENT: AskWeather
PRED INTENT: AskWeather (conf=0.955)
---


C:\Users\25718\AppData\Local\Temp\ipykernel_62452\1608082985.py:17: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  "input_ids": torch.tensor(self.encodings["input_ids"][idx]),
C:\Users\25718\AppData\Local\Temp\ipykernel_62452\1608082985.py:18: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  "attention_mask": torch.tensor(self.encodings["attention_mask"][idx]),


TOKENS     : ['[CLS]', 'will', 'it', 'rain', 'in', 'london', 'tomorrow', '[SEP]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]']
TRUE SLOTS : ['-', 'O', 'O', 'O', 'O', 'B-CITY', 'B-DAY', '-', '-', '-', '-', '-', '-']
PRED SLOTS : ['-', 'O', 'O', 'O', 'O', 'B-CITY', 'B-DAY', '-', '-', '-', '-', '-', '-']
TRUE INTENT: AskWeather
PRED INTENT: AskWeather
PRED INTENT: AskWeather (conf=0.965)
---

Intent: Goodbye
TOKENS     : ['[CLS]', 'see', 'you', 'soon', '[SEP]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]']
TRUE SLOTS : ['-', 'O', 'O', 'O', '-', '-', '-', '-', '-', '-', '-', '-', '-']
PRED SLOTS : ['-', 'O', 'O', 'O', '-', '-', '-', '-', '-', '-', '-', '-', '-']
TRUE INTENT: Goodbye
PRED INTENT: Goodbye
PRED INTENT: Goodbye (conf=0.920)
---
TOKENS     : ['[CLS]', 'see', 'you', '[SEP]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]']
TRUE SLOTS : ['-', 'O', 'O', '-', '-', '-', '-', '-', '-', '-', '-', '-', '-']
PRED SLOTS : ['-', 'O

**Step 10 - Temporal Resolution**

In this notebook, we are not fulfilling/achieving any intent, but the next step in the Voice Assistant pipeline would be fulfilling, meaning actually performing the action.  For doing so, slot information must be disambiguated.  For example, temporal resolution (also called date/time resolution) must be performed to be able to actually set an alarm.

***

***TODO: Program temporal resolution.***

- The Set_Alarm intent contains 2 slots for day and time.  Use that information to perform temporal resolution.
- (hint) the *dateparser* python library will help you out.
- Show a few sentences with the slot found and the transformation to the absolute standardized date/time.
- If no day is provided, make sure you use the default (e.g. set the alarm for 5am - no day, but you have a default) to perform temporal resolution.
***

In [40]:
!pip install dateparser


[notice] A new release of pip is available: 25.1.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [41]:
import dateparser
from datetime import datetime

def predict_and_resolve(sentence, model, tokenizer, id2intent, id2slot, device):
    """Run a sentence through the model and resolve temporal expressions."""
    model.eval()

    # Tokenize the sentence (split into words first for proper alignment)
    words = sentence.split()
    encoding = tokenizer(words, is_split_into_words=True, return_tensors="pt",
                         padding=True, truncation=True, max_length=20)
    input_ids = encoding["input_ids"].to(device)
    attention_mask = encoding["attention_mask"].to(device)

    with torch.no_grad():
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)

    # Intent prediction with confidence
    intent_probs = torch.softmax(outputs["intent_logits"], dim=1)
    intent_idx = torch.argmax(intent_probs, dim=1).item()
    intent = id2intent[intent_idx]
    confidence = intent_probs[0, intent_idx].item()

    # Slot extraction: group B- and I- tags into slot values
    slot_preds = torch.argmax(outputs["slot_logits"], dim=2)[0]
    word_ids = encoding.word_ids(batch_index=0)

    current_slot_type = None
    current_tokens = []
    extracted_slots = {}

    prev_word_id = None
    for i, word_id in enumerate(word_ids):
        if word_id is None or word_id == prev_word_id:
            prev_word_id = word_id
            continue

        slot = id2slot[slot_preds[i].item()]

        if slot.startswith("B-"):
            if current_slot_type:
                extracted_slots[current_slot_type] = " ".join(current_tokens)
            current_slot_type = slot[2:]
            current_tokens = [words[word_id]]
        elif slot.startswith("I-") and current_slot_type == slot[2:]:
            current_tokens.append(words[word_id])
        else:
            if current_slot_type:
                extracted_slots[current_slot_type] = " ".join(current_tokens)
                current_slot_type = None
                current_tokens = []

        prev_word_id = word_id

    if current_slot_type:
        extracted_slots[current_slot_type] = " ".join(current_tokens)

    # Display prediction results
    print(f"Sentence : {sentence}")
    print(f"Intent   : {intent} (confidence={confidence:.3f})")
    print(f"Slots    : {extracted_slots}")

    # Temporal resolution for SetAlarm
    if intent == "SetAlarm":
        time_value = extracted_slots.get("TIME", "")
        day_value = extracted_slots.get("DAY", "")
        name_value = extracted_slots.get("NAME", "default")

        # Combine day and time for dateparser
        if day_value and time_value:
            temporal_str = f"{day_value} {time_value}"
        elif time_value:
            temporal_str = time_value
        elif day_value:
            temporal_str = day_value
        else:
            temporal_str = "7 am"  # default if nothing detected
            print(f"  No time detected, using default: {temporal_str}")

        resolved = dateparser.parse(temporal_str)
        if resolved:
            print(f"  Alarm name : {name_value}")
            print(f"  Raw temporal: '{temporal_str}'")
            print(f"  Resolved to : {resolved.strftime('%Y-%m-%d %H:%M:%S')}")
        else:
            print(f"  Could not resolve temporal expression: '{temporal_str}'")
    print()

In [42]:
test_sentences = [
    "set an alarm for 7 am",                          # TIME only, no day -> default today
    "wake me at 6 am tomorrow",                       # TIME + DAY
    "set the work alarm for 5 am next Monday",        # TIME + NAME + DAY
    "set alarm for 8 am on Friday",                   # TIME + DAY
    "set the morning alarm for 6 am",                 # TIME + NAME, no day -> default today
    "set an alarm at 9 am on Saturday",               # TIME + DAY
    "wake me up at 5 am next Sunday",                 # TIME + DAY
]

print("=" * 60)
print("TEMPORAL RESOLUTION EXAMPLES")
print("=" * 60)
print()
for s in test_sentences:
    predict_and_resolve(s, model, tokenizer, id2intent, id2slot, device)

TEMPORAL RESOLUTION EXAMPLES

Sentence : set an alarm for 7 am
Intent   : SetAlarm (confidence=0.972)
Slots    : {'TIME': '7 am'}
  Alarm name : default
  Raw temporal: '7 am'
  Resolved to : 2026-03-05 07:00:00

Sentence : wake me at 6 am tomorrow
Intent   : SetAlarm (confidence=0.968)
Slots    : {'TIME': '6 am', 'DAY': 'tomorrow'}
  Alarm name : default
  Raw temporal: 'tomorrow 6 am'
  Resolved to : 2026-03-05 06:00:00

Sentence : set the work alarm for 5 am next Monday
Intent   : SetAlarm (confidence=0.973)
Slots    : {'NAME': 'work', 'TIME': '5 am', 'DAY': 'next Monday'}
  Could not resolve temporal expression: 'next Monday 5 am'

Sentence : set alarm for 8 am on Friday
Intent   : SetAlarm (confidence=0.973)
Slots    : {'TIME': '8 am', 'DAY': 'Friday'}
  Alarm name : default
  Raw temporal: 'Friday 8 am'
  Resolved to : 2026-02-27 08:00:00

Sentence : set the morning alarm for 6 am
Intent   : SetAlarm (confidence=0.974)
Slots    : {'NAME': 'morning', 'TIME': '6 am'}
  Alarm name :